# Lab 3: Isolate — Keeping Contexts Separate\n\n**Concept:** Isolation prevents cross-contamination between contexts. Each user, domain, or test run gets its own clean context so information doesn't leak between queries.\n\n## Exercises\n- 3a. Create per-user keyword profiles that prune the same document differently (from `naive_regex`).\n- 3b. Build per-domain TF-IDF indexes and route queries to the correct one (from `semantic_chunk`).\n- 3c. Run the same question through all 3 pruning methods in isolated contexts and compare answers (from `pruning_dashboard` & `llmlingua`).\n

## Setup\n\n```bash\npip install openai scikit-learn numpy\nexport OPENROUTER_API_KEY='sk-or-v1-...'\n```

In [ ]:
import os, re, json, pickle, glob\nimport numpy as np\nfrom openai import OpenAI\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.metrics.pairwise import cosine_similarity\n\nclient = OpenAI(api_key=os.getenv('OPENROUTER_API_KEY'), base_url='https://openrouter.ai/api/v1')

## Exercises

In [ ]:
import os, re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENROUTER_API_KEY"), base_url="https://openrouter.ai/api/v1")

document = """The company reported strong Q3 earnings. Revenue grew by 15% year over year to $45.2 million.
The expansion of the Frankfurt server farm cost $14.73 million. This was partially offset by a $2.1 million
green energy tax rebate. Employee headcount remained stable at 1,200."""
sentences = re.split(r'(?<=[.!?])\s+', document)

# === 3a: Per-user profiles ===
def prune_for_user(question, extra_keywords):
    kw = {w.lower() for w in re.findall(r'\b[a-zA-Z]{3,}\b', question)} | {k.lower() for k in extra_keywords}
    return ' '.join(s for s in sentences if any(k in s.lower() for k in kw)) or ' '.join(sentences[:1])

print("=== 3a: User Isolation ===")
for role, extras in [("CFO", ["revenue", "cost", "margin"]),
                     ("HR", ["headcount", "employee"])]:
    print(f"  {role}: {prune_for_user('What happened?', extras)[:80]}...")

# === 3b: Per-domain indexes ===
domains = {
    "finance": [s for s in sentences if any(w in s for w in ["revenue", "cost", "million"])],
    "staffing": [s for s in sentences if any(w in s for w in ["headcount", "employee"])],
}
indexes = {}
for name, docs in domains.items():
    v = TfidfVectorizer()
    indexes[name] = {"vect": v.fit(docs), "docs": docs}

def route_query(q):
    kw = q.lower().split()
    domain = "finance" if any(k in ["revenue", "cost", "capex"] for k in kw) else "staffing"
    idx = indexes[domain]
    qv = idx["vect"].transform([q])
    scores = cosine_similarity(qv, idx["vect"].transform(idx["docs"])).flatten()
    return domain, [idx["docs"][i] for i in np.argsort(scores)[::-1][:1]]

print("\n=== 3b: Domain Routing ===")
for q in ["What was the revenue?", "How many employees?"]:
    dom, results = route_query(q)
    print(f"  '{q}' -> {dom}: {results[0][:50] if results else 'none'}")

# === 3c: Isolated method comparison ===
question = "What was the CAPEX and what offset it?"
contexts = {}

keywords = {w.lower() for w in re.findall(r'\b[a-zA-Z]{3,}\b', question)}
contexts["Naive Regex"] = ' '.join(s for s in sentences if any(k in s.lower() for k in keywords))

vect = TfidfVectorizer(ngram_range=(1,2))
vect.fit([question] + sentences)
scores = cosine_similarity(vect.transform([question]), vect.transform(sentences)).flatten()
contexts["Semantic"] = '\n\n'.join(sentences[i] for i in np.argsort(scores)[::-1][:2])

resp = client.chat.completions.create(
    model="meta-llama/llama-3.3-70b-instruct:free",
    messages=[{"role": "user", "content":
        f"Compress this to only what answers: {question}\n\nDocument: {document}"}],
)
contexts["LLM-Compressed"] = resp.choices[0].message.content

print("\n=== 3c: Isolated Method Comparison ===")
for name, ctx in contexts.items():
    r = client.chat.completions.create(
        model="meta-llama/llama-3.3-70b-instruct:free",
        messages=[{"role": "user", "content": f"Context: {ctx}\n\n{question}"}],
        temperature=0,
    )
    print(f"  {name} ({len(ctx.split())}w): {r.choices[0].message.content[:80]}...")

## Summary\n\nIsolation prevents cross-contamination between contexts. Each user, domain, or test run gets its own clean context so information doesn't leak between queries.